一、未提供少量示例的提示词模板

In [17]:
import os
from typing import final

import dotenv
from langchain_core.messages import HumanMessage,AIMessage
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate,FewShotChatMessagePromptTemplate,ChatPromptTemplate
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY1")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL")

llm = ChatOpenAI(
    model='gpt-4o-mini',
    temperature=0.8,
    max_tokens=20
)

# res=llm.invoke('2#1是多少')
# print(res.content)

二、提供少量示例的提示词模板

使用FewShotPromptTemplate

In [12]:
#创建PromptTemplates实例
example_prompt = PromptTemplate.from_template(
    template='input:{input}\noutput:{output}',
)

#提供一些示例
examples = [
    {'input': '北京天气怎么样', 'output': '北京市'},
    {'input': '南京下雨吗', 'output': '南京市'},
    {'input': '武汉热吗', 'output': '武汉市'},
]

few_shot_prompt_template = FewShotPromptTemplate(
    example_prompt=example_prompt,
    suffix='input:{input}\noutput:',  #声明在示例后面的提示词模板
    examples=examples,
    input_variables=['input']
)

prompt = few_shot_prompt_template.invoke(input={'input': '天津今天多少度'})
res = llm.invoke(prompt)
print(res.content)


天津市


与Chat一起使用 FewShotChatMessagePromptTemplate

In [16]:
from langchain_core.prompts import MessagesPlaceholder

example_prompt = ChatPromptTemplate.from_template(
    template='input:{input}\noutput:{output}'
)

examples = [
    {'input': '1ff2', 'output': '1'},
    {'input': '2ff2', 'output': '4', },
    {'input': '3ff2', 'output': '9', },
]

few_shot_prompt_template = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
    input_variables=['input']
)

final_prompt = ChatPromptTemplate.from_messages(
    [
        ('system','你是一个数学天才'),
        few_shot_prompt_template,
        MessagesPlaceholder(variable_name='msgs')
    ]
)

prompt_res = few_shot_prompt_template.invoke(input={'input': '4ff2','msgs':HumanMessage(content='哈哈')})
res = llm.invoke(prompt_res)
print(res.content)

It looks like you are providing inputs in hexadecimal format with a particular output pattern. To decipher the output


In [23]:
from langchain_core.prompts import MessagesPlaceholder

# 1. 定义示例
examples = [
    {
        "input": "1ff2",
        "output": "1"  # 1² = 1
    },
    {
        "input": "2ff2",
        "output": "4"  # 2² = 4
    },
    {
        "input": "3ff2",
        "output": "9"  # 3² = 9
    }
]

# 2. 创建少样本提示模板
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=ChatPromptTemplate.from_messages([
        ("human", "{input}"),
        ("ai", "{output}")
    ]),
    examples=examples
)

# 3. 构建最终提示词
final_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个数学天才，能从示例中学习运算规律并应用。"),
    few_shot_prompt,
    ("human", "{input}")
])

# 4. 生成提示词并调用模型
user_input = "4ff2"
prompt = final_prompt.invoke({"input": user_input})

print("=== 生成的提示词 ===")
for message in prompt.messages:
    print(f"{message.type}: {message.content}")

# 调用模型
res = llm.invoke(prompt)
print(res.content)

=== 生成的提示词 ===
system: 你是一个数学天才，能从示例中学习运算规律并应用。
human: 1ff2
ai: 1
human: 2ff2
ai: 4
human: 3ff2
ai: 9
human: 4ff2
16


三、Example Selector 示例选择器(暂时不需要了解)